In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.gold;


In [0]:
CREATE OR REPLACE TABLE workspace.gold.city_overview AS
SELECT
    city,
    COUNT(*) AS total_listings,
    COUNT(DISTINCT host_id) AS total_hosts,

    ROUND(AVG(price), 2) AS avg_price_eur,
    ROUND(percentile_approx(price, 0.5), 2)
        AS median_price_eur,

    ROUND(AVG(availability_365), 1)
        AS avg_availability_days,

    ROUND(AVG(review_scores_rating), 2)
        AS avg_review_rating,

    COUNT(review_scores_rating)
        AS listings_with_rating

FROM workspace.silver.listings
WHERE city IN ('rome', 'sicily')
GROUP BY city;


num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM workspace.gold.city_overview
ORDER BY city;


city,total_listings,total_hosts,avg_price_eur,median_price_eur,avg_availability_days,avg_review_rating,listings_with_rating
rome,37451,19779,254.34,188.5,207.4,4.77,32990
sicily,56783,29025,172.51,115.0,227.0,4.75,42708


In [0]:
CREATE OR REPLACE TABLE workspace.gold.room_type_metrics AS
WITH room_stats AS (
    SELECT
        city,
        room_type,

        COUNT(*) AS total_listings,
        COUNT(DISTINCT host_id) AS total_hosts,

        ROUND(AVG(price), 2) AS avg_price_eur,
        ROUND(percentile_approx(price, 0.5), 2)
            AS median_price_eur,

        ROUND(AVG(availability_365), 1)
            AS avg_availability_days,

        ROUND(AVG(review_scores_rating), 2)
            AS avg_review_rating

    FROM workspace.silver.listings
    WHERE city IN ('rome', 'sicily')
      AND room_type IS NOT NULL
    GROUP BY city, room_type
)

SELECT
    *,
    ROUND(
        100.0 * total_listings
        / SUM(total_listings) OVER (PARTITION BY city),
        2
    ) AS listing_share_pct
FROM room_stats;


num_affected_rows,num_inserted_rows


In [0]:
SELECT
    city,
    room_type,
    total_listings,
    median_price_eur,
    listing_share_pct
FROM workspace.gold.room_type_metrics
ORDER BY city, total_listings DESC;


city,room_type,total_listings,median_price_eur,listing_share_pct
rome,Entire home/apt,27886,205.98,74.46
rome,Private room,9266,139.0,24.74
rome,Hotel room,200,203.0,0.53
rome,Shared room,99,49.0,0.26
sicily,Entire home/apt,45151,123.0,79.51
sicily,Private room,11455,91.29,20.17
sicily,Hotel room,110,119.0,0.19
sicily,Shared room,67,33.0,0.12


In [0]:
DESCRIBE TABLE workspace.silver.listings;

col_name,data_type,comment
city,string,null
snapshot_date,date,null
id,bigint,null
host_id,bigint,null
latitude,double,null
longitude,double,null
accommodates,double,null
bedrooms,double,null
beds,double,null
minimum_nights,double,null


In [0]:

CREATE OR REPLACE TABLE workspace.gold.neighbourhood_metrics AS
SELECT
    city,
    neighbourhood_cleansed AS neighbourhood,

    COUNT(*) AS total_listings,
    COUNT(DISTINCT host_id) AS total_hosts,

    ROUND(AVG(price), 2) AS avg_price_eur,
    ROUND(percentile_approx(price, 0.5), 2)
        AS median_price_eur,

    ROUND(AVG(availability_365), 1)
        AS avg_availability_days,

    ROUND(AVG(review_scores_rating), 2)
        AS avg_review_rating

FROM workspace.silver.listings
WHERE city IN ('rome', 'sicily')
  AND neighbourhood_cleansed IS NOT NULL
  AND TRIM(neighbourhood_cleansed) <> ''
GROUP BY city, neighbourhood_cleansed;


num_affected_rows,num_inserted_rows


In [0]:

SELECT
    city,
    neighbourhood,
    total_listings,
    median_price_eur
FROM workspace.gold.neighbourhood_metrics
WHERE total_listings >= 30
ORDER BY city, median_price_eur DESC;


city,neighbourhood,total_listings,median_price_eur
rome,I Centro Storico,19119,236.0
rome,XIII Aurelia,2649,174.0
rome,II Parioli/Nomentano,2518,171.25
rome,XII Monte Verde,2004,156.9
rome,VII San Giovanni/Cinecittà,3179,150.0
rome,VIII Appia Antica,1178,144.94
rome,XIV Monte Mario,792,143.0
rome,XI Arvalia/Portuense,751,135.5
rome,XV Cassia/Flaminia,564,135.0
rome,III Monte Sacro,598,128.42


In [0]:

SELECT
    city,
    COUNT(*) AS total_neighbourhoods,
    SUM(total_listings) AS total_listings
FROM workspace.gold.neighbourhood_metrics
GROUP BY city
ORDER BY city;


city,total_neighbourhoods,total_listings
rome,15,37451
sicily,376,56783
